# Apartment Price Regression — Case Study

**Objective.** Build an interpretable regression model for apartment unit-price (`price_per_sqm`).
Total transaction price is reconstructed as a secondary business diagnostic.

**Approach.** EDA → feature engineering motivated by EDA findings → candidate model comparison on a test set → holdout evaluation with bootstrap uncertainty and residual analysis.

**Key constraint.** The holdout set is separated *before* any exploratory analysis or modeling decision. All EDA, feature engineering, and model selection use the development partition only.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import statsmodels.formula.api as smf
from scipy import stats

RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# ── Plotting defaults ──
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

## 1. Data loading and global audit

In [ ]:
df_raw = pd.read_csv("apartment_price_regression_case.csv")

display(df_raw.head())
print("Shape:", df_raw.shape)

In [ ]:
display(df_raw.select_dtypes(include=np.number).describe().T)

In [ ]:
display(df_raw.select_dtypes(include=["object", "category", "bool"]).describe().T)

The raw file contains **5,218 rows × 22 columns**. Variables split naturally into four groups: location (`city`, `district`, `district_quality`), apartment characteristics (`surface_m2`, `rooms`, `bedrooms`, `floor`, amenities), building/environment (`noise_level_db`, `distance_metro_m`, `building_age_years`, etc.) and the target `price_eur`.

Key first observations: surface runs 16–185 sqm (median 57.6), median total price ~€311k, dominant city exposure is Paris (1,609 rows / 31%). Several columns already show missing counts (`noise_level_db`, `building_age_years`, `distance_school_m`, `balcony_m2`). The unique `apartment_id` count (5,200) vs row count (5,218) signals 18 likely duplicates.

## 2. Target construction, deduplication and protected holdout

In [ ]:
df = df_raw.drop_duplicates().copy()
df["price_per_sqm"] = df["price_eur"] / df["surface_m2"]

# ── Convert categoricals to string (avoids bool/str mix issues in SimpleImputer) ──
cat_cols_raw = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_cols_raw:
    df[col] = df[col].astype(str)

print("Raw shape:", df_raw.shape)
print("Post-dedup shape:", df.shape)
print("Duplicates removed:", len(df_raw) - len(df))

18 exact duplicates are dropped, leaving **5,200 clean rows**.

**Why `price_per_sqm` as target?** In real-estate valuation, the unit price isolates the *quality premium* of a location/apartment from the mechanical relationship between price and size. A model predicting total price would be dominated by the surface × unit-price interaction. Predicting `price_per_sqm` lets the model focus on what actually varies: location, condition, amenities. Total price is reconstructed afterwards as `predicted_ppsqm × surface`.

In [ ]:
dev_df, holdout_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("Dev set: ", dev_df.shape)
print("Holdout: ", holdout_df.shape)

**1,040 apartments** are set aside as a protected holdout. The remaining **4,160 rows** (the development set) are used for all EDA, feature engineering, and model selection. No modeling decision below uses holdout data.

## 3. EDA — development sample only

### 3.1 Sanity checks

In [ ]:
expected_bounds = {
    "surface_m2": (10, 250),
    "rooms": (1, 10),
    "bedrooms": (0, 8),
    "floor": (0, 40),
    "total_floors": (1, 50),
    "price_per_sqm": (2_000, 25_000),
}

sanity_rows = []
for col, (lo, hi) in expected_bounds.items():
    sanity_rows.append({
        "column": col,
        "min": dev_df[col].min(),
        "max": dev_df[col].max(),
        "below_expected": int((dev_df[col] < lo).sum()),
        "above_expected": int((dev_df[col] > hi).sum()),
    })

display(pd.DataFrame(sanity_rows))

No structural anomalies for surface, room counts, or floor variables. Only **4 observations** fall below €2,000/sqm — they remain close to the threshold, so they are kept. No upper-bound violations.

This is reassuring: the dataset is clean enough that aggressive winsorisation is unnecessary.

### 3.2 Target distribution and size effects

In [ ]:
target_summary = dev_df[["price_eur", "surface_m2", "price_per_sqm"]].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T

display(target_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

axes[0].hist(dev_df["price_per_sqm"], bins=50, edgecolor="white", linewidth=0.3)
axes[0].set_title("price_per_sqm distribution")
axes[0].set_xlabel("EUR / sqm")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

axes[1].hist(dev_df["price_eur"], bins=50, edgecolor="white", linewidth=0.3)
axes[1].set_title("Total price distribution")
axes[1].set_xlabel("EUR")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

axes[2].hist(dev_df["surface_m2"], bins=50, edgecolor="white", linewidth=0.3)
axes[2].set_title("Surface distribution")
axes[2].set_xlabel("sqm")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(dev_df["surface_m2"], dev_df["price_per_sqm"], alpha=0.2, s=12)
axes[0].set_title("Surface vs unit price")
axes[0].set_xlabel("Surface (sqm)")
axes[0].set_ylabel("EUR / sqm")

axes[1].scatter(dev_df["surface_m2"], dev_df["price_eur"], alpha=0.2, s=12)
axes[1].set_title("Surface vs total price")
axes[1].set_xlabel("Surface (sqm)")
axes[1].set_ylabel("EUR")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.show()

Median unit price: **€4,984/sqm**, mean **€6,354/sqm**, std **€3,490/sqm**. The distribution is right-skewed (mean >> median), driven by high-value Parisian apartments.

The scatter plots validate the target choice: total price increases almost mechanically with surface (left), while `price_per_sqm` isolates a different signal — with a mild *negative* slope (larger apartments tend to have lower unit prices, consistent with the well-known "small flat premium" in urban markets). This confirms that predicting unit price, then multiplying by surface, is the right modelling strategy.

**EDA takeaway → feature engineering:** the mild negative slope suggests `surface_m2` should be included as a feature even though the target already normalises by surface. A `log(surface)` or interaction might capture the nonlinearity.

### 3.3 Missing values — volume and price impact

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": dev_df.isna().sum(),
    "missing_pct": dev_df.isna().mean().mul(100).round(2),
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

display(missing_summary)

missing_cols = missing_summary.index.tolist()

In [ ]:
# ── Core question: do apartments with missing data have different unit prices? ──
price_impact_rows = []
for col in missing_cols:
    mask = dev_df[col].isna()
    price_missing = dev_df.loc[mask, "price_per_sqm"]
    price_present = dev_df.loc[~mask, "price_per_sqm"]
    
    # Welch t-test (unequal variances)
    t_stat, p_val = stats.ttest_ind(price_missing, price_present, equal_var=False)
    
    price_impact_rows.append({
        "feature": col,
        "n_missing": mask.sum(),
        "mean_ppsqm_missing": price_missing.mean(),
        "mean_ppsqm_present": price_present.mean(),
        "delta_ppsqm": price_missing.mean() - price_present.mean(),
        "delta_pct": (price_missing.mean() - price_present.mean()) / price_present.mean() * 100,
        "t_stat": t_stat,
        "p_value": p_val,
    })

price_impact = pd.DataFrame(price_impact_rows).sort_values("delta_ppsqm", key=abs, ascending=False)
for c in ["mean_ppsqm_missing", "mean_ppsqm_present", "delta_ppsqm"]:
    price_impact[c] = price_impact[c].round(0).astype(int)
price_impact["delta_pct"] = price_impact["delta_pct"].round(1)
price_impact["p_value"] = price_impact["p_value"].map(lambda x: f"{x:.4f}")

display(price_impact)

The table above is the key diagnostic: for each feature with missing values, we compare the mean `price_per_sqm` of apartments where that feature is missing vs present.

`noise_level_db` stands out: apartments where noise data is missing are **+€830/sqm more expensive** on average (p < 0.001). This makes economic sense — high-end listings in prime Parisian districts are more likely to lack noise measurements. `balcony_m2` and `building_age_years` show the opposite pattern: missing values are associated with **lower** unit prices (−€474 and −€455/sqm respectively), with `building_age_years` statistically significant (p = 0.028) and `balcony_m2` borderline (p = 0.053). `distance_school_m`, `heating_type`, and `energy_rating` show small, non-significant gaps.

The picture is more nuanced than "missing = expensive": the direction and magnitude of the gap varies by feature. This reinforces the case for **missingness indicator flags** — they let the model learn feature-specific missing-vs-present effects rather than applying a single imputation strategy blindly.

The next question is whether these gaps persist *within* price segments, or whether they are purely compositional (e.g. more `noise_level_db` missing in Paris → higher average price mechanically).

In [ ]:
# ── Does the missing/present price gap persist within price segments? ──
#    If it vanishes within segments, the gap is purely compositional.
#    If it persists, missingness carries *additional* information beyond location/price tier.

price_buckets = pd.qcut(dev_df["price_per_sqm"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])

within_segment_rows = []
for col in missing_cols:
    mask = dev_df[col].isna()
    for bucket in ["Q1_low", "Q2", "Q3", "Q4_high"]:
        in_bucket = price_buckets == bucket
        n_miss = (mask & in_bucket).sum()
        n_pres = (~mask & in_bucket).sum()
        if n_miss >= 10:  # only compute if enough observations
            mean_miss = dev_df.loc[mask & in_bucket, "price_per_sqm"].mean()
            mean_pres = dev_df.loc[~mask & in_bucket, "price_per_sqm"].mean()
            within_segment_rows.append({
                "feature": col,
                "price_bucket": bucket,
                "n_missing": n_miss,
                "mean_ppsqm_missing": round(mean_miss),
                "mean_ppsqm_present": round(mean_pres),
                "delta_ppsqm": round(mean_miss - mean_pres),
            })

within_df = pd.DataFrame(within_segment_rows)
if len(within_df):
    display(within_df.pivot_table(
        index="feature", columns="price_bucket", values="delta_ppsqm", aggfunc="first"
    ).reindex(columns=["Q1_low", "Q2", "Q3", "Q4_high"]))

In [ ]:
# ── Visual: missing rate by district quality and condition ──
missing_profile = dev_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

missing_profile.groupby("district_quality", observed=False)[missing_cols].apply(
    lambda x: x.isna().mean()
).mul(100).plot(kind="bar", ax=axes[0])
axes[0].set_title("Missingness by district quality")
axes[0].set_ylabel("Missing rate (%)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(fontsize=7)

missing_profile.groupby("property_condition", observed=False)[missing_cols].apply(
    lambda x: x.isna().mean()
).mul(100).plot(kind="bar", ax=axes[1])
axes[1].set_title("Missingness by condition")
axes[1].set_ylabel("Missing rate (%)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

The within-segment pivot reveals an important result. For `noise_level_db` — the feature with the largest aggregate gap (+€830) — the within-quartile deltas collapse to near-zero (Q1: +€35, Q2: −€30, Q3: +€45, Q4: −€5). This means the aggregate gap is **almost entirely compositional**: more noise data is missing in expensive Parisian areas, which mechanically inflates the missing-group average. Within a given price tier, knowing that noise data is missing tells you almost nothing extra about price.

The other features show noisy, inconsistent within-segment patterns — some positive in one quartile, negative in another — with no clear residual signal after conditioning on price level.

This is a useful finding for modelling: the missingness indicators will likely contribute less incremental signal than the aggregate gaps suggest, because the location dummies already capture most of the composition effect. Still, including them is cheap insurance — the model can learn to ignore them if they add no value, and they protect against edge cases where missingness interacts with features the location dummies don't fully capture.

**EDA takeaway → preprocessing:** Median imputation + missingness indicator flags for all numerical columns. Categorical missingness (`energy_rating`, `heating_type`) encoded as an explicit `Missing` level.

### 3.4 Price gradients by categorical segment

In [ ]:
categorical_cols_for_eda = ["city", "district_quality", "property_condition", "energy_rating", "heating_type"]

for col in categorical_cols_for_eda:
    segment = (
        dev_df.groupby(col, dropna=False)
        .agg(
            count=("price_per_sqm", "size"),
            median_ppsqm=("price_per_sqm", "median"),
            mean_ppsqm=("price_per_sqm", "mean"),
            std_ppsqm=("price_per_sqm", "std"),
            median_surface=("surface_m2", "median"),
        )
        .sort_values("median_ppsqm", ascending=False)
    )
    print(f"\n── {col} ──")
    display(segment)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

dev_df.boxplot(column="price_per_sqm", by="city", ax=axes[0], rot=45)
axes[0].set_title("Unit price by city")
axes[0].set_xlabel("")
axes[0].set_ylabel("EUR / sqm")

dev_df.boxplot(column="price_per_sqm", by="district_quality", ax=axes[1], rot=45)
axes[1].set_title("Unit price by district quality")
axes[1].set_xlabel("")
axes[1].set_ylabel("EUR / sqm")

plt.suptitle("")
plt.tight_layout()
plt.show()

City is the dominant driver: **Paris median €10,521/sqm** — roughly 2× Lyon (€5,774) and 3× Toulouse/Lille/Nantes/Marseille (~€3,300–3,850). District quality adds a second layer: **prime ≈ €8,265/sqm** vs **outer ≈ €3,656/sqm**.

Condition and energy rating provide additional, monotonic signal: new > renovated > good > average > to_renovate, and A/B > C > D > E > F > G. Missing energy rating sits mid-range (median €5,143), between C and D — confirming the preprocessing choice to encode it as its own category rather than force-imputing.

The within-group standard deviation for Paris (~€2,500) is much higher than for Toulouse (~€700), which foreshadows heteroscedastic residuals in the high-price segment.

**EDA takeaway → modelling:** The categorical price gradients are very large (>€5,000 spread across cities). Linear models with one-hot encoding will capture most of this. Tree-based models should capture it implicitly through splits. The key question is whether nonlinear interactions (e.g. `city × district_quality × condition`) add meaningful signal beyond main effects.

### 3.5 Numerical correlations and nonlinearities

In [ ]:
numeric_cols = [
    "surface_m2", "rooms", "bedrooms", "floor", "total_floors", "balcony_m2",
    "noise_level_db", "distance_metro_m", "distance_school_m", "green_space_distance_m",
    "building_age_years", "price_per_sqm",
]

corr = dev_df[numeric_cols].corr(numeric_only=True)["price_per_sqm"].drop("price_per_sqm").sort_values()
display(corr.to_frame("corr_with_ppsqm"))

In [ ]:
# ── Check for nonlinearity in the top numeric features ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# axes = axes.ravel() if several rows

for ax, col in zip(axes, ["distance_metro_m", "noise_level_db", "building_age_years"]):
    ax.scatter(dev_df[col], dev_df["price_per_sqm"], alpha=0.15, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel("price_per_sqm")
    ax.set_title(f"r = {dev_df[[col, 'price_per_sqm']].corr().iloc[0,1]:.3f}")

plt.tight_layout()
plt.show()

Numerical correlations are weaker than the categorical gradients — expected in a location-dominated problem. The strongest are `noise_level_db` (+0.26 — noisier = more central = more expensive) and `distance_metro_m` (−0.23 — further from metro = cheaper).

The scatter plots reveal **nonlinearity**: `distance_metro_m` shows a concave decay that `log(distance)` would linearise. `noise_level_db` has a positive slope that is really a proxy for centrality. `building_age_years` shows weak signal with a possible U-shape (very old heritage buildings can be valuable).

**EDA takeaway → feature engineering:**
- `log(distance_metro_m)` and `log(green_space_distance_m)` to linearise distance decay
- `floor / total_floors` ratio (higher relative floor → premium view)
- `has_balcony` binary + `amenity_count` composite
- Keep `district` as a categorical (140 levels): the tree-based model can exploit it, while Ridge will regularise it

## 4. Feature engineering and significance

This section engineers new features motivated by EDA findings, defines the feature set used throughout modelling, and runs a simplified OLS with robust standard errors to assess statistical significance before moving to ML models.

### 4.1 Engineered features

All features below are motivated by the EDA findings above:
- **`floor_ratio`** ← Section 3.5: higher relative floor should capture view/noise premium
- **`has_balcony`**, **`amenity_count`** ← Section 3.4: condition/amenity gradients are visible
- **`log_distance_metro`**, **`log_green_space_distance`** ← Section 3.5: distance decay is concave, log linearises it

In [ ]:
def add_features(data):
    out = data.copy()
    out["floor_ratio"] = out["floor"] / out["total_floors"].replace(0, np.nan)
    out["has_balcony"] = (out["balcony_m2"].fillna(0) > 0).astype(int)
    out["amenity_count"] = out[["elevator", "terrace", "parking", "has_balcony"]].sum(axis=1)
    out["log_distance_metro"] = np.log1p(out["distance_metro_m"])
    out["log_green_space_distance"] = np.log1p(out["green_space_distance_m"])
    return out

dev_fe = add_features(dev_df)
holdout_fe = add_features(holdout_df)

In [ ]:
target = "price_per_sqm"
price_target = "price_eur"

excluded_cols = ["apartment_id", "price_eur", "price_per_sqm"]
features = [col for col in dev_fe.columns if col not in excluded_cols]

numeric_features = dev_fe[features].select_dtypes(include=np.number).columns.tolist()
categorical_features = dev_fe[features].select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Features: {len(features)} total — {len(numeric_features)} numeric, {len(categorical_features)} categorical")

### 4.2 Simplified OLS with robust standard errors

In [ ]:
ols_df = dev_fe.copy()

# Impute categoricals with explicit "Missing" level
for col in categorical_features:
    ols_df[col] = ols_df[col].fillna("Missing")

# Impute numericals with median
for col in numeric_features:
    if ols_df[col].isna().any():
        ols_df[col] = ols_df[col].fillna(ols_df[col].median())

ols_formula = (
    target + " ~ " + " + ".join(
        numeric_features + [f"C({c})" for c in categorical_features]
    )
)

print("OLS formula:")
print(ols_formula)

Before choosing a covariance estimator, we test for heteroscedasticity on the classical OLS residuals. If the null of constant variance is rejected, HC3 robust SEs are justified.

In [ ]:
# Heteroscedasticity diagnostic 

from statsmodels.stats.diagnostic import het_breuschpagan, het_white

# ── Step 1: fit OLS with classical (non-robust) SEs ──
ols_classical = smf.ols(formula=ols_formula, data=ols_df).fit()  # default: non-robust

# ── Step 2: visual diagnostic — OLS residuals vs fitted values ──
fitted = ols_classical.fittedvalues
resid  = ols_classical.resid
exog   = ols_classical.model.exog

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(fitted, resid, alpha=0.15, s=10, color="steelblue")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_xlabel("Fitted values (€/sqm)")
axes[0].set_ylabel("Residuals (€/sqm)")
axes[0].set_title("OLS residuals vs fitted values")

axes[1].scatter(fitted, np.abs(resid), alpha=0.15, s=10, color="indianred")
axes[1].set_xlabel("Fitted values (€/sqm)")
axes[1].set_ylabel("|Residuals| (€/sqm)")
axes[1].set_title("Absolute residuals vs fitted (spread diagnostic)")

plt.tight_layout()
plt.show()

# ── Step 3: formal tests on classical residuals ──

# Breusch-Pagan (tests if residual variance is a function of regressors)
bp_stat, bp_p, bp_f, bp_fp = het_breuschpagan(resid, exog)

# White test (adds squares and cross-products — more general)
white_stat, white_p, white_f, white_fp = het_white(resid, exog)

print("── Heteroscedasticity tests on OLS residuals ──\n")
print(f"Breusch-Pagan  LM = {bp_stat:10.2f}   p = {bp_p:.4e}")
print(f"White          LM = {white_stat:10.2f}   p = {white_p:.4e}")

The residual-vs-fitted plot shows a clear megaphone pattern: dispersion roughly doubles between €4,000/sqm and €12,000/sqm. The Breusch-Pagan (p ≈ 10⁻¹³⁸) and White (p ≈ 10⁻¹¹²) tests confirm formally — homoscedasticity is decisively rejected.

Beyond the variance issue, the left panel also reveals structure in the residuals: the cloud bends rather than scattering symmetrically around zero, suggesting nonlinearity that the linear specification does not capture. This is consistent with the role of this OLS — a simplified interpretive benchmark, not the predictive model.

Both patterns have the same economic root: the Parisian premium segment concentrates more within-segment variability (arrondissement, building prestige, renovation quality) and more complex price dynamics than provincial markets. Under heteroscedasticity, classical standard errors are biased, so we refit below with HC3 robust SEs.

In [ ]:
ols_model = smf.ols(formula=ols_formula, data=ols_df).fit(cov_type="HC3")
print(ols_model.summary())

The OLS is not used for prediction — it serves as a statistical sanity check with proper inference. Key observations:

**R² = 0.918** — most of the signal is captured by main effects alone. This establishes a strong linear baseline; the question of how much nonlinear models improve upon it will be answered in §6.

All city effects are highly significant (p < 0.001) with HC3 robust SEs. Paris: +€5,304/sqm (t ≈ 100), confirming the dominant location effect. Surface: −€10.5/sqm per additional sqm (the "small flat premium"). `log_distance_metro`: −€210/sqm (metro proximity premium). `noise_level_db`: −€36/sqm per dB, which appears contradictory to the positive EDA correlation (+0.26) — but makes sense conditional on location: *within* a city and district, more noise reduces value; the positive unconditional correlation was confounded by centrality.

Energy rating shows a monotonic gradient: B (−€111, n.s.), C (−€328), D (−€484), E (−€690), F (−€889), G (−€1,148), all relative to A. The "Missing" category coefficient (−€592) sits between D and E, consistent with the §3.3 finding that missing energy data is not concentrated at either extreme.

The condition number is large, indicating collinearity among the location dummies. However, HC3 standard errors are robust to heteroscedasticity only — not to spatial clustering. Apartments in the same district share unobservables (school quality, street-level amenities, micro-location). The next cell refits with SEs clustered by district to test whether the inference changes.

In [ ]:
# ── Refit same OLS with SEs clustered by district ──
#    If apartments in the same district share unobservables (school quality,
#    street-level amenities, micro-location effects), HC3 SEs are anticonservative.
#    Clustering absorbs within-district residual correlation.

ols_clustered = smf.ols(formula=ols_formula, data=ols_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": ols_df["district"]},
)

# ── Compare p-values: HC3 vs clustered ──
comparison = pd.DataFrame({
    "coef": ols_model.params,
    "p_hc3": ols_model.pvalues,
    "p_clustered": ols_clustered.pvalues,
}).round(4)

comparison["significance_lost"] = (comparison["p_hc3"] < 0.05) & (comparison["p_clustered"] >= 0.05)

print("── Coefficients where clustering changes significance (p < 0.05 → p ≥ 0.05) ──")
lost = comparison[comparison["significance_lost"]]
if len(lost):
    display(lost)
else:
    print("None — all coefficients that were significant under HC3 remain significant under clustering.")

print("\n── Full comparison (selected coefficients) ──")
show_vars = [v for v in comparison.index if any(k in v for k in ["surface", "noise", "floor", "amenity", "building_age", "log_dist"])]
display(comparison.loc[show_vars])

The clustered SEs (140 district groups) do not overturn any HC3 result: **all coefficients that were significant under HC3 remain significant under clustering**. The main categorical effects are completely unaffected (p ≈ 0.000 in both cases), which is expected — their variation is primarily between districts.

The most informative comparison is `building_age_years`: its p-value moves from 0.043 (HC3) to 0.045 (clustered) — still just below 5%, but the margin is razor-thin. In practice, this coefficient (−€1.0/year) should be interpreted with caution: it is the weakest signal in the model, and a slightly different clustering level (e.g. by city × district_quality) could push it past the threshold. All other numerical variables (`surface_m2`, `log_distance_metro`, `noise_level_db`, `floor_ratio`, `amenity_count`) are robust to clustering with p ≈ 0.000.

This is a reassuring result: the HC3 inference was not materially anticonservative for this specification and clustering level.

## 5. Preprocessing and candidate models

In [ ]:
# ── Explicit train/test split (75/25 of dev set) ──
train_df, test_df = train_test_split(
    dev_fe,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]
y_test_price = test_df[price_target]

X_holdout = holdout_fe[features]
y_holdout = holdout_fe[target]
y_holdout_price = holdout_fe[price_target]

print("Train:  ", X_train.shape)
print("Test:   ", X_test.shape)
print("Holdout:", X_holdout.shape)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

**Preprocessing rationale:**
- Numerical: median imputation (robust to skew) + missingness indicator flags (captures the MAR patterns found in §3.3) + standardisation (needed for Ridge/ElasticNet, harmless for trees).
- Categorical: explicit `Missing` level (preserves the informative-missingness signal) + one-hot encoding with `handle_unknown="ignore"` for holdout safety.

**Candidate models** span two linear and two tree-based families to test whether nonlinear interactions improve over the strong main-effect signal found in EDA:

In [ ]:
model_specs = {
    "ridge": {
        "model": Ridge(),
        "param_grid": {
            # Alpha controls L2 strength. Range covers weak to moderate regularisation.
            # With ~190 OHE features, alpha=3-30 is a reasonable range.
            "model__alpha": [3.0, 10.0, 30.0],
        },
    },
    "elastic_net": {
        "model": ElasticNet(max_iter=10_000, random_state=RANDOM_STATE),
        "param_grid": {
            # Alpha: overall penalty strength; l1_ratio: sparsity vs ridge trade-off.
            # Low alpha + high l1_ratio = aggressive feature selection.
            "model__alpha": [0.003, 0.01, 0.03],
            "model__l1_ratio": [0.2, 0.5, 0.8],
        },
    },
    "random_forest": {
        "model": RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=1),
        "param_grid": {
            # max_depth=None vs 16 tests deep vs moderate trees.
            # min_samples_leaf=5/12 controls overfitting. max_features=0.7 decorrelates trees.
            "model__max_depth": [None, 16],
            "model__min_samples_leaf": [5, 12],
            "model__max_features": [0.7],
        },
    },
    "hist_gradient_boosting": {
        "model": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "param_grid": {
            # lr=0.05/0.1: lower lr needs more iterations but generalises better.
            # max_leaf_nodes=15/31: controls tree complexity; 31 is sklearn default.
            # max_iter=200/350: with lr=0.05, 200 iterations may underfit; 350 gives headroom.
            "model__learning_rate": [0.05, 0.1],
            "model__max_iter": [200, 350],
            "model__max_leaf_nodes": [15, 31],
            "model__l2_regularization": [0.0],
        },
    },
}

## 6. Candidate model training and test-set evaluation

### 6.1 GridSearchCV on train, evaluate best on test

In [ ]:
best_family_estimators = {}
results_rows = []

for name, spec in model_specs.items():
    print(f"Fitting {name}...")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", spec["model"]),
    ])
    gs = GridSearchCV(
        pipe,
        param_grid=spec["param_grid"],
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    best = gs.best_estimator_
    best_family_estimators[name] = best
    best_idx = gs.best_index_

    pred_train = best.predict(X_train)
    pred_test = best.predict(X_test)

    results_rows.append({
        "model": name,
        "best_params": gs.best_params_,
        "train_mae": round(mean_absolute_error(y_train, pred_train), 1),
        "cv_mae": round(-gs.cv_results_["mean_test_score"][best_idx], 1),
        "cv_mae_std": round(gs.cv_results_["std_test_score"][best_idx], 1),
        "test_mae": round(mean_absolute_error(y_test, pred_test), 1),
        "train_r2": round(r2_score(y_train, pred_train), 4),
        "test_r2": round(r2_score(y_test, pred_test), 4),
    })

model_results = pd.DataFrame(results_rows).sort_values("test_mae").reset_index(drop=True)
target_std_test = y_test.std()
model_results["mae_over_std"] = (model_results["test_mae"] / target_std_test).round(4)

display(model_results)
print(f"\nTest target std: €{target_std_test:.0f}/sqm")

**HistGradientBoosting wins clearly.** Best config achieves **MAE ≈ €535/sqm**, **RMSE ≈ €765/sqm**, **R² ≈ 0.951** on test. The top ~8 rows are all boosting variants (MAE €535–571), followed by RF (~€611–614), then Ridge/ElasticNet (~€635). The MAE/σ ratio of ~15.4% is reasonable for a cross-sectional real-estate model.

The linear models perform surprisingly well (only ~19% worse than the best), which confirms the EDA finding: main categorical effects dominate. The tree-based improvement captures nonlinear interactions (e.g. city × district, condition × energy) beyond one-hot main effects.

Comparing with the OLS baseline from §4.2 (R² = 0.918), the gap to the best boosting model (R² ≈ 0.951) quantifies the added value of nonlinear interactions: roughly **3.3 pp of R²**. This is meaningful but not dramatic — consistent with a problem dominated by strong main effects where the marginal gains from tree-based flexibility are incremental.

### 6.2 Naive benchmark and empirical prediction band

In [ ]:
best_row = model_results.iloc[0]
best_model_name = best_row["model"]
best_model_on_train = best_family_estimators[best_model_name]

best_test_pred = best_model_on_train.predict(X_test)
best_test_abs_errors = np.abs(y_test.values - best_test_pred)

naive_pred = np.full(len(y_test), y_train.mean())
naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = mean_squared_error(y_test, naive_pred) ** 0.5

print(f"Naive baseline (train mean): MAE = €{naive_mae:.0f}/sqm, RMSE = €{naive_rmse:.0f}/sqm")
print(f"Selected model ({best_model_name}): MAE = €{best_row['test_mae']:.0f}/sqm")
print(f"Relative improvement: {1 - best_row['test_mae'] / naive_mae:.0%}")
print(f"\n95th-percentile absolute error: €{np.percentile(best_test_abs_errors, 95):.0f}/sqm (model)")
print(f"95th-percentile absolute error: €{np.percentile(np.abs(y_test.values - naive_pred), 95):.0f}/sqm (naive)")

The naive baseline (predict train mean for every apartment) yields **MAE ≈ €2,870/sqm**. The selected model reduces this by **~81%** — an enormous relative improvement, but expected given how much the target varies across cities.

The 95th-percentile absolute error gives a rough individual-prediction "uncertainty band": **±€1,594/sqm** for the model vs ±€6,905 for the naive predictor. This is *not* a formal confidence interval — it is an empirical error quantile. For a proper uncertainty estimate, see the bootstrap analysis in §9.

## 7. Pre-holdout model interpretation

### 7.1 Permutation importance (model-agnostic)

In [ ]:
# ── Permutation importance on test set (model-agnostic, avoids the bias of
#    impurity-based importance toward high-cardinality categoricals) ──
perm_result = permutation_importance(
    best_model_on_train, X_test, y_test,
    n_repeats=15,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
)

perm_imp = pd.DataFrame({
    "feature": features,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

display(perm_imp.head(20))

Permutation importance on the test set is preferable to impurity-based RF importance because it avoids the well-known bias toward high-cardinality features (like `district` with 140 levels). It measures the actual increase in MAE when each feature is randomly shuffled.

`city` dominates by a wide margin (importance ~2,598), followed by `district_quality` (~637) and `property_condition` (~210). Notably, `district` (140 levels) drops to a very low permutation importance (~3.9) despite ranking much higher in the RF impurity-based metric — a textbook illustration of the high-cardinality bias in impurity-based measures. `surface_m2` (~87) and `energy_rating` (~74) contribute meaningfully, followed by `noise_level_db` (~38) and `distance_metro_m` (~31).

The engineered `log_distance_metro` and `has_balcony` features show zero permutation importance, confirming that the tree-based model already captures the nonlinear distance effect and the balcony signal through `distance_metro_m` and `balcony_m2` directly. These features add value for the linear models but are redundant for the boosting model.

### 7.2 Random Forest impurity-based importance (comparison)

In [ ]:
def get_feature_names_from_preprocessor(fitted_preprocessor):
    return fitted_preprocessor.get_feature_names_out()

rf_estimator = best_family_estimators["random_forest"]
rf_feature_names = get_feature_names_from_preprocessor(rf_estimator.named_steps["preprocessor"])
rf_importances = rf_estimator.named_steps["model"].feature_importances_

rf_importance = (
    pd.DataFrame({"feature": rf_feature_names, "importance": rf_importances})
    .sort_values("importance", ascending=False)
)

# ── Group by raw feature ──
def map_to_raw_feature(transformed_feature):
    if transformed_feature.startswith("num__"):
        raw = transformed_feature.replace("num__", "").replace("missingindicator_", "")
        return raw
    if transformed_feature.startswith("cat__"):
        remainder = transformed_feature.replace("cat__", "")
        for col in sorted(categorical_features, key=len, reverse=True):
            if remainder.startswith(col + "_"):
                return col
        return remainder
    return transformed_feature

rf_importance["raw_feature"] = rf_importance["feature"].map(map_to_raw_feature)
rf_grouped = (
    rf_importance.groupby("raw_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

display(rf_grouped.head(15))

The impurity-based importance tells a similar story at the top — `city` dominates (~0.78), then `district_quality` (~0.09) — but `district` appears at ~0.04, far above its permutation importance of ~0.004. This 10× inflation is a textbook example of the high-cardinality bias: with 140 levels, `district` gets many splitting opportunities in random forest trees regardless of true predictive value. The permutation importance provides a more honest comparison. Both methods agree that location categoricals >>> condition >>> numerical features, but they disagree on `district`'s contribution.

### 7.3 Ridge coefficients — interpretable linear benchmark

In [ ]:
ridge_estimator = best_family_estimators["ridge"]
ridge_feature_names = get_feature_names_from_preprocessor(ridge_estimator.named_steps["preprocessor"])
ridge_coefs = ridge_estimator.named_steps["model"].coef_

ridge_coef_table = (
    pd.DataFrame({
        "feature": ridge_feature_names,
        "coef_ppsqm": ridge_coefs,
        "abs_coef": np.abs(ridge_coefs),
    })
    .sort_values("abs_coef", ascending=False)
)

print("── Top 15 positive coefficients ──")
display(ridge_coef_table.sort_values("coef_ppsqm", ascending=False).head(15))

print("\n── Top 15 negative coefficients ──")
display(ridge_coef_table.sort_values("coef_ppsqm", ascending=True).head(15))

The Ridge coefficients are directionally coherent with EDA:
- **Paris: +€4,781/sqm**, prime district: +€1,759/sqm — the largest effects, consistent with §3.4
- **Specific arrondissements** add on top: PAR-01 (+€1,833), PAR-02 (+€1,373), PAR-03 (+€1,094) — capturing within-Paris micro-location effects
- **New/renovated condition: +€868/+€593/sqm** — monotonic condition gradient as expected
- **Energy A/B: ~+€438–440/sqm** — the DPE premium, though A and B are nearly identical
- **Negative: Toulouse (−€1,570), Lille (−€1,463), outer district (−€1,360), to_renovate (−€958)** — all consistent

These are conditional associations in a regularised linear model, not causal effects. But the interpretability is useful for model sanity-checking: if Paris had a *negative* coefficient, something would be wrong.

## 8. Final holdout evaluation

### 8.1 Retrain on full dev set, evaluate on holdout

In [ ]:
# ── Refit best model on full dev set (train + test) before holdout evaluation ──
X_dev_full = dev_fe[features]
y_dev_full = dev_fe[target]

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", type(best_model_on_train.named_steps["model"])(**best_model_on_train.named_steps["model"].get_params())),
])
final_model.fit(X_dev_full, y_dev_full)

holdout_pred_ppsqm = final_model.predict(X_holdout)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "holdout_mae_ppsqm": mean_absolute_error(y_holdout, holdout_pred_ppsqm),
    "holdout_rmse_ppsqm": mean_squared_error(y_holdout, holdout_pred_ppsqm) ** 0.5,
    "holdout_r2_ppsqm": r2_score(y_holdout, holdout_pred_ppsqm),
    "target_std_holdout": y_holdout.std(),
}])

holdout_metrics["mae_over_std"] = holdout_metrics["holdout_mae_ppsqm"] / holdout_metrics["target_std_holdout"]

display(holdout_metrics)

print(f"\nTest MAE:     €{best_row['test_mae']:.0f}/sqm")
print(f"Holdout MAE: €{holdout_metrics['holdout_mae_ppsqm'].iloc[0]:.0f}/sqm")
print(f"Degradation: {(holdout_metrics['holdout_mae_ppsqm'].iloc[0] - best_row['test_mae']) / best_row['test_mae']:.1%}")

Holdout: **MAE = €570/sqm, RMSE = €819/sqm, R² = 0.948**. The MAE/σ ratio (15.9%) is consistent with the test ratio (15.4%), and the degradation from test to holdout is only **~6.5%** — a healthy sign that the test-based selection did not materially overfit.

### 8.2 Reconstructed total price (secondary business metric)

In [ ]:
holdout_pred_price = holdout_pred_ppsqm * X_holdout["surface_m2"]

price_metrics = pd.DataFrame([{
    "holdout_mae_eur": mean_absolute_error(y_holdout_price, holdout_pred_price),
    "holdout_rmse_eur": mean_squared_error(y_holdout_price, holdout_pred_price) ** 0.5,
    "holdout_mape": np.mean(np.abs((y_holdout_price - holdout_pred_price) / y_holdout_price)),
}])

display(price_metrics)

Reconstructed total-price error: **MAE = €35.1k, MAPE = 8.9%**. For a non-production exploratory model, a sub-10% MAPE on transaction price is a reasonable baseline. The RMSE (€54.6k) is higher due to heavy tails in the high-price segment.

## 9. Holdout diagnostics and uncertainty

### 9.1 Predicted vs actual + residual analysis

In [ ]:
holdout_diag = pd.DataFrame({
    "actual_ppsqm": y_holdout.values,
    "predicted_ppsqm": holdout_pred_ppsqm,
    "actual_eur": y_holdout_price.values,
    "predicted_eur": holdout_pred_price.values,
    "surface_m2": X_holdout["surface_m2"].values,
    "city": X_holdout["city"].values,
    "district_quality": X_holdout["district_quality"].values,
})

holdout_diag["residual_ppsqm"] = holdout_diag["actual_ppsqm"] - holdout_diag["predicted_ppsqm"]
holdout_diag["abs_error_ppsqm"] = holdout_diag["residual_ppsqm"].abs()
holdout_diag["abs_error_eur"] = np.abs(holdout_diag["actual_eur"] - holdout_diag["predicted_eur"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Predicted vs actual
ax = axes[0]
ax.scatter(holdout_diag["actual_ppsqm"], holdout_diag["predicted_ppsqm"], alpha=0.3, s=15)
lims = [holdout_diag["actual_ppsqm"].min(), holdout_diag["actual_ppsqm"].max()]
ax.plot(lims, lims, "--", color="grey", linewidth=1)
ax.set_title("Predicted vs actual (ppsqm)")
ax.set_xlabel("Actual EUR/sqm")
ax.set_ylabel("Predicted EUR/sqm")

# Residual vs predicted (heteroscedasticity check)
ax = axes[1]
ax.scatter(holdout_diag["predicted_ppsqm"], holdout_diag["residual_ppsqm"], alpha=0.3, s=15)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Residual vs predicted")
ax.set_xlabel("Predicted EUR/sqm")
ax.set_ylabel("Residual EUR/sqm")

# Residual distribution
ax = axes[2]
ax.hist(holdout_diag["residual_ppsqm"], bins=50, edgecolor="white", linewidth=0.3)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Residual distribution")
ax.set_xlabel("Residual EUR/sqm")

plt.tight_layout()
plt.show()

The predicted-vs-actual plot hugs the diagonal with visible fan-out at the high end. The residual-vs-predicted plot confirms **heteroscedasticity**: the model is less precise for expensive apartments (predicted ppsqm > €10k). This is expected — the Parisian premium-segment has more within-segment variance (unique heritage buildings, specific arrondissements, exceptional views). The residual distribution is approximately symmetric but heavy-tailed, consistent with the heteroscedasticity.

This has practical implications: a constant prediction interval would be too wide for cheap apartments and too narrow for expensive ones. A production model should use quantile regression or conditional variance estimation.

### 9.2 Error decomposition by segment

In [ ]:
holdout_diag["surface_bucket"] = pd.qcut(
    holdout_diag["surface_m2"], q=4, labels=["small", "mid_small", "mid_large", "large"]
)
holdout_diag["ppsqm_bucket"] = pd.qcut(
    holdout_diag["actual_ppsqm"], q=4, labels=["low", "mid_low", "mid_high", "high"]
)

print("── Error by surface quartile ──")
display(
    holdout_diag.groupby("surface_bucket", observed=False)
    .agg(n=("abs_error_ppsqm", "size"), mae_ppsqm=("abs_error_ppsqm", "mean"),
         mae_eur=("abs_error_eur", "mean"), median_surface=("surface_m2", "median"))
)

print("\n── Error by price quartile ──")
display(
    holdout_diag.groupby("ppsqm_bucket", observed=False)
    .agg(n=("abs_error_ppsqm", "size"), mae_ppsqm=("abs_error_ppsqm", "mean"),
         mae_eur=("abs_error_eur", "mean"), median_ppsqm=("actual_ppsqm", "median"))
)

print("\n── Error by city ──")
display(
    holdout_diag.groupby("city", observed=False)
    .agg(n=("abs_error_ppsqm", "size"), mae_ppsqm=("abs_error_ppsqm", "mean"),
         mae_eur=("abs_error_eur", "mean"), median_ppsqm=("actual_ppsqm", "median"))
    .sort_values("mae_ppsqm", ascending=False)
)

The error decomposition reveals the **main residual risk**:
- **By price quartile:** low/mid-low segments have MAE ~€308–344/sqm, but the **high segment reaches €1,056/sqm** — 3× worse. This confirms the heteroscedasticity.
- **By surface:** MAE in ppsqm is relatively stable (~€493–648), but in EUR terms, large apartments naturally generate larger transaction errors (€55k vs €20k).
- **By city:** Paris has by far the highest MAE (**€995/sqm**, with €61k in total-price terms), consistent with its higher within-city variance. Provincial cities are much tighter: Toulouse (€288), Marseille (€304), Lille (€304), Nantes (€351).

A next-step improvement should focus on the premium segment — possibly through Paris-specific sub-models, richer arrondissement-level features, or quantile regression for segment-conditional uncertainty.

### 9.3 Bootstrap confidence intervals on holdout metrics

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
B = 2_000
n = len(y_holdout)
idx = np.arange(n)

actual_ppsqm_arr = holdout_diag["actual_ppsqm"].to_numpy()
pred_ppsqm_arr = holdout_diag["predicted_ppsqm"].to_numpy()
actual_eur_arr = holdout_diag["actual_eur"].to_numpy()
pred_eur_arr = holdout_diag["predicted_eur"].to_numpy()

boot_rows = []
for _ in range(B):
    s = rng.choice(idx, size=n, replace=True)
    boot_rows.append({
        "mae_ppsqm": mean_absolute_error(actual_ppsqm_arr[s], pred_ppsqm_arr[s]),
        "rmse_ppsqm": mean_squared_error(actual_ppsqm_arr[s], pred_ppsqm_arr[s]) ** 0.5,
        "r2_ppsqm": r2_score(actual_ppsqm_arr[s], pred_ppsqm_arr[s]),
        "mae_eur": mean_absolute_error(actual_eur_arr[s], pred_eur_arr[s]),
    })

boot_df = pd.DataFrame(boot_rows)
boot_summary = boot_df.quantile([0.025, 0.5, 0.975]).T
boot_summary.columns = ["ci_2.5%", "median", "ci_97.5%"]
display(boot_summary)

The 95% bootstrap intervals (2,000 resamples) are tight:
- **MAE ppsqm:** €535–605/sqm (median €570)
- **R² ppsqm:** 0.941–0.954 (median 0.948)
- **MAE EUR:** €32.6k–37.6k (median €35.1k)

The intervals are narrow enough to confidently rank the model above the naive baseline (MAE ~€2,870). The R² interval excludes the OLS-only R² (0.918), confirming that the nonlinear model adds statistically significant predictive power beyond main effects.

**Note:** These bootstrap CIs quantify *sampling uncertainty in the metric estimate on this holdout*, not prediction uncertainty for a new apartment. They answer "how precisely do we know this model's MAE?" — not "what's the error on a single prediction?".

### 9.4 Permutation test — is the model significantly better than chance?

In [ ]:
# Shuffle the target on holdout and compute MAE to build a null distribution
n_perm = 500
null_maes = []
actual_mae = mean_absolute_error(actual_ppsqm_arr, pred_ppsqm_arr)

for _ in range(n_perm):
    shuffled = rng.permutation(actual_ppsqm_arr)
    null_maes.append(mean_absolute_error(shuffled, pred_ppsqm_arr))

null_maes = np.array(null_maes)
p_value = (null_maes <= actual_mae).mean()

print(f"Actual holdout MAE:  €{actual_mae:.0f}/sqm")
print(f"Null MAE (mean):     €{null_maes.mean():.0f}/sqm")
print(f"Null MAE (min):      €{null_maes.min():.0f}/sqm")
print(f"Permutation p-value: {p_value:.4f}")
print(f"\nThe model's MAE is lower than all {n_perm} null permutations → p < {1/n_perm:.4f}.")

The permutation test shuffles the actual holdout targets and recomputes MAE 500 times. The model's MAE (€570) is lower than *every* null permutation (null mean MAE = €3,780, null minimum = €3,565) — the p-value is 0.000, formally p < 0.002.

This is a coarse test (it would reject almost any reasonable model on this problem given the enormous gap between model and null), but it closes the statistical hygiene loop. The more informative question is whether the *improvement from OLS to boosting* is significant — the bootstrap R² intervals above address this: the 95% CI for R² (0.941–0.954) excludes the OLS R² of 0.918.

## 10. Final remarks

### Summary

The final model is a **HistGradientBoosting regressor** trained on `price_per_sqm` with 25 features (19 numerical + 6 categorical). It achieves **MAE = €570/sqm (R² = 0.948)** on a protected holdout, reducing the naive-mean baseline error by ~81%. Bootstrap 95% CI for MAE: [€535, €605]. Permutation test: p < 0.002.

### What the model captures well

Location dominates: `city` + `district_quality` + `property_condition` account for ~90% of permutation importance. The model correctly prices the Paris premium (~€10,500/sqm), the condition gradient (new > renovated > good > average > to_renovate), the energy rating monotonicity (A > ... > G), and the metro proximity premium. These are economically interpretable and directionally robust across Ridge, RF, and boosting.

### Main limitations

1. **Heteroscedastic residuals.** MAE in the top price quartile (€1,056/sqm) is 3× worse than in the bottom quartile (€308/sqm). Paris alone has MAE = €995/sqm. The model is less precise where it matters most commercially. This is a structural feature of real-estate pricing, not a modelling bug — premium apartments have more idiosyncratic variance (views, renovation quality, exact street).

2. **Clustering robustness checked, not exhaustive.** The OLS was refitted with SEs clustered by district (§4.2): all significant coefficients survived, though `building_age_years` remains borderline (p = 0.045). A finer clustering level (building or street) or a spatial error model with coordinates would be a stricter test.

3. **Cross-sectional only.** No temporal dimension is available, so we cannot test whether the model's coefficients are stable across market cycles or whether recent price trends leak into the cross-section.

4. **Missingness signal is mostly compositional.** The §3.3 within-segment analysis showed that the large aggregate gap for `noise_level_db` (missing = +€830/sqm) collapses to near-zero within price quartiles. The missingness indicators are cheap insurance, but they likely add limited incremental signal beyond what location dummies already capture.

### Concrete next steps

1. **Premium-segment calibration:** Fit a separate sub-model or use quantile regression for the top price quartile (Paris prime / central), where MAE is 3× the average.
2. **Spatial features:** District-level median income, transaction density, or coordinates → spatial kernel features to reduce the within-Paris residual variance.
3. **Conditional prediction intervals:** Replace the constant ±€1,594 band with a quantile regression or conformal prediction interval that widens for expensive apartments.
4. **Temporal validation:** If timestamped data becomes available, use walk-forward validation to test temporal stability.